# 133. Clone Graph
**Difficulty:** 🟡 Medium · **Topic:** Graph · **LeetCode:** https://leetcode.com/problems/clone-graph/

## 💡 Concepts

**Core concept(s):** Traverse the graph (**DFS** or **BFS**) while keeping a **map from each old node to its copy**.

**Why it applies here:** To copy a graph you must copy every node once and rewire the copies to each other — not to the originals. A hash map "original → copy" lets you reuse a copy when you meet a node again, which also stops you looping forever on cycles.

**Key intuition:** Copy a node the first time you see it, remember that copy, and connect copies to copies.

---

### 📚 What is a Graph?
A **graph** is dots (**nodes/vertices**) joined by lines (**edges**). Edges can be **directed** (one-way, like prerequisites) or **undirected** (two-way, like friendships). A **grid** is just a graph where each cell links to its neighbors.
- **In Python:** usually an **adjacency list** — a `dict` mapping each node to the list of nodes it connects to.

### 📚 What is DFS (Depth-First Search)?
**DFS** follows one path as deep as it goes, then backtracks. On graphs you must remember **visited** nodes so you don't loop forever.
- **Complexity:** **O(V + E)** — each node and edge once.
- **In Python:** recursion or an explicit stack, plus a `visited` set.

### 📚 What is BFS (Breadth-First Search)?
**BFS** explores in rings outward from the start using a **queue**, visiting nearer nodes first.
- **Complexity:** **O(V + E)**; great for shortest number of steps.
- **In Python:** `collections.deque` plus a `visited` set.

### 📚 What is a Hash Set?
A `set` answers "is x here?" in **O(1)** average. Here it lets us test membership (e.g. "is x-1 present?") without scanning.

---

**Prerequisite knowledge:**
- Adjacency (each node lists its neighbors).
- A visited/clone map.

## 📝 Problem

Given a reference to a node in a connected undirected graph, return a **deep copy** (all new nodes, same connections).

> Two approaches, both `O(V + E)`: DFS and BFS.

In [ ]:
from collections import deque

class Node:
    """A graph node: a value plus a list of the nodes it connects to."""
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []

### Approach 1 — DFS with a Clone Map

**Idea:** Recurse. The first time you see a node, make its copy and store it; then copy its neighbors (reusing stored copies on repeats).

**Time:** `O(V + E)`. **Space:** `O(V)`.

In [ ]:
def clone_dfs(node):
    if not node:
        return None
    old_to_new = {}                        # original node -> its freshly-made copy
    def dfs(n):
        if n in old_to_new:                # already copied this node...
            return old_to_new[n]           # ...reuse the copy (this also stops cycles looping)
        copy = Node(n.val)                 # make the copy BEFORE recursing (so cycles work)
        old_to_new[n] = copy
        for nb in n.neighbors:             # copy each neighbor and link copy -> copy
            copy.neighbors.append(dfs(nb))
        return copy
    return dfs(node)

### Approach 2 — BFS with a Queue

**Idea:** Copy the start, then BFS: for each node, ensure each neighbor has a copy and wire the copies together.

**Time:** `O(V + E)`. **Space:** `O(V)`.

In [ ]:
def clone_bfs(node):
    if not node:
        return None
    old_to_new = {node: Node(node.val)}    # copy the start node first
    q = deque([node])
    while q:
        cur = q.popleft()
        for nb in cur.neighbors:
            if nb not in old_to_new:       # first time we see this neighbor...
                old_to_new[nb] = Node(nb.val)  # ...make its copy and queue it
                q.append(nb)
            old_to_new[cur].neighbors.append(old_to_new[nb])  # link copy -> copy
    return old_to_new[node]

In [ ]:
# Correctness check: clone is a deep copy with identical structure
def build_sample():
    a, b, c, d = Node(1), Node(2), Node(3), Node(4)
    a.neighbors = [b, d]; b.neighbors = [a, c]; c.neighbors = [b, d]; d.neighbors = [a, c]
    return a

def signature(node):
    seen = {}
    def dfs(n):
        if n in seen: return
        seen[n] = sorted(x.val for x in n.neighbors)
        for x in n.neighbors: dfs(x)
    dfs(node)
    return sorted((n.val, tuple(v)) for n, v in seen.items())

for clone in (clone_dfs, clone_bfs):
    orig = build_sample()
    cp = clone(orig)
    assert cp is not orig, "must be a new object"
    assert signature(orig) == signature(cp), "structure differs"
    print(clone.__name__, "OK")
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)` / `O(V+E)` | ≈ **2×** |
| `O(n log n)`      | ≈ **2×** (slightly more) |
| `O(n²)`           | ≈ **4×** |

Inputs are shaped to force the worst case while keeping recursion shallow (stars / checkerboards) so nothing overflows the stack.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def _star(n):
    nodes = [Node(i) for i in range(n)]
    for i in range(1, n):
        nodes[0].neighbors.append(nodes[i]); nodes[i].neighbors.append(nodes[0])
    return nodes[0]

def make_worst_case(n):
    return (_star(n),)   # star: shallow DFS, V nodes + ~V edges
solutions = {
    "dfs O(V+E)": clone_dfs,
    "bfs O(V+E)": clone_bfs,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Visited/clone map:** the universal way to traverse a graph without looping and to remember work (here, the copies).
- **DFS or BFS — same cost:** both visit every node and edge once.
- **Signal:** "deep copy a graph", "traverse with cycles".
- **Related problems:** Number of Islands, Course Schedule, Copy List with Random Pointer.
- **Common pitfalls:** (1) wiring copies to originals; (2) no visited map → infinite loop on cycles.